Conditional Chain

In [1]:
from  langchain_openai import ChatOpenAI
import os 
from dotenv import load_dotenv

# This function will load all the variables from the .env file and will 
# make them available in the os.environ dictionary (env variables)
load_dotenv() 

if os.environ.get("OPENAI_API_KEY"):
    print("Bro API KEY Variable exists")
else:
    raise ValueError("OPENAI_API_KEY not found")

from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from  langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, PydanticOutputParser

llm_openai = ChatOpenAI(model="gpt-5-mini",temperature=0)

Bro API KEY Variable exists


In [2]:
from pydantic import BaseModel
from typing import Literal

class llm_schema(BaseModel):
    movie_summary_flag: Literal["positive", "negative"]



CHAIN WITH Conditional Chains

In [3]:
# TASK -1 [Prompt]

prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are a movie review evaluator"),
    ("human", "Please categorize the movie review as positive or negative : {input}")])

In [4]:
# TASK - 2 [LLM]

llm_structured_output = llm_openai.with_structured_output(llm_schema)

In [5]:
# TASK - 3 [Custom Runnable]
from langchain_core.runnables import RunnableLambda

def pydantic_json(input:llm_schema)-> str:

    return input.model_dump()['movie_summary_flag']

pydantic_json_lambda = RunnableLambda(pydantic_json)

Conditional Chain 1

In [6]:
# TASK - 1 [Prompt]

movie_reviewer_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert movie critic with years of experience reviewing films. Your goal is to provide a balanced review that highlights both the strengths and weaknesses of a movie, analyzing the plot, acting, and cinematography. Be fair but critical."),
    ("human", "Write a comprehensive review for the following movie: {movie_name}")
])

# TASK - 2 [LLM]

llm_openai = ChatOpenAI(model="gpt-5-mini",temperature=0)

# TASK - 3 [Str Parser]

str_parser = StrOutputParser()

movie_reviewer = movie_reviewer_prompt | llm_openai | str_parser

In [7]:



movie_reviewer.invoke('Sholay')

'Sholay (1975) — Directed by Ramesh Sippy; written by Salim–Javed; produced by G. P. Sippy. Starring Amitabh Bachchan, Dharmendra, Sanjeev Kumar, Hema Malini, Jaya Bhaduri, Amjad Khan. Music by R. D. Burman. Cinematography: Dwarka Divecha. Runtime: ~204 minutes.\n\nOverview / Plot\nSholay is best described as an ambitious mash-up: part revenge drama, part buddy movie, part western, and thoroughly masala. The story centers on a retired police officer, Thakur Baldev Singh (Sanjeev Kumar), who hires two small‑time but charismatic criminals—Jai (Amitabh Bachchan) and Veeru (Dharmendra)—to capture the ruthless dacoit Gabbar Singh (Amjad Khan) terrorizing the village of Ramgarh. Layers of humor, romance, and tragedy are woven in as the villagers, the two heroes, and Thakur collide with Gabbar’s brutality, culminating in a violent, emotionally charged conflict.\n\nStrengths\n\n- Writing and Dialogues: Salim–Javed’s screenplay is lean in its central revenge arc yet luxuriant in memorable scene

In [8]:
def movie_chain(text:dict):

    text = text["text"]

    # TASK - 1 [Prompt]
    movie_reviewer_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert movie critic with years of experience reviewing films. Your goal is to provide a balanced review that highlights both the strengths and weaknesses of a movie, analyzing the plot, acting, and cinematography. Be fair but critical."),
    ("human", "Write a comprehensive review for the following movie: {movie_name}")
])
    
    # TASK - 2 [LLM]
    llm_openai = ChatOpenAI(model="gpt-5-mini",temperature=0)

    # TASK - 3 [Str Parser]
    str_parser = StrOutputParser()

    chain_insta = movie_reviewer_prompt | llm_openai | str_parser

    result = chain_insta.invoke(text)

    return result

movie_chain_runnable = RunnableLambda(movie_chain)

Final Orchestration

In [10]:
from langchain_core.runnables import RunnableBranch
conditional_chain = RunnableBranch(
    (lambda x: "positive" in x, movie_reviewer),
     movie_chain_runnable
)

final_orchestrator = prompt_template | llm_structured_output | pydantic_json_lambda | conditional_chain

In [11]:
final_orchestrator.invoke({"input": "I loved this KGF movie"})

'I’m not sure if you mean a film actually titled “Positive,” or you want a review that’s positive in tone. Could you clarify?\n\nOptions:\n- If you mean a real movie called “Positive,” tell me the year, director, or a link (there are several films/shorts with that title).\n- If you want a positive-toned review of a specific film, tell me the film’s title.\n- If you’d like, I can write a comprehensive, positive review of an imagined movie called “Positive.”\n\nTell me which you want and any specifics (tone, length, focus on plot/acting/cinematography), and I’ll write it.'